# 9.2 그래디언트 소실·폭발, 활성함수, 초기화 — 실습 노트북

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SuminHan/book-ml/blob/main/notebooks/ml1/chapter09_2_vanishing_gradients.ipynb)

책 본문: [9.2 그래디언트 소실·폭발, 활성함수, 초기화](https://smhanlab.com/book-ml/kor/ml1/chapter09/2.html)

이 노트북은 책 9.2절의 모든 수치를 **실제로 실행해서** 검증합니다:

1. 10층 시그모이드 망에서 그래디언트 노름이 출력층 → 입력층으로 **553×** 줄어드는지(순전파+역전파)
2. 초기화 표준편차 σ=1.0 / 0.01 / He일 때 10개 ReLU 층 통과 후 활성값 분산이 **폭발/소실/안정**으로 갈라지는지
3. **Dying ReLU**: 같은 학습률에서 ReLU는 죽은 뉴런이 97%까지 치솟아 손실이 고착되고,
   Leaky ReLU는 계속 수렴하는지
4. 활성함수 6종(sigmoid, tanh, ReLU, Leaky, GELU, Swish)과 미분을 그림으로 비교
5. 7층 XOR: tanh+Xavier(폭발 → NaN) vs ReLU+He(정체 → 0.25) vs Adam(수렴)
6. Xavier vs He: "2"가 ReLU의 ½ 절단을 보상한다는 것의 의미


In [1]:
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

matplotlib.rcParams["font.sans-serif"] = ["Noto Sans CJK KR", "NanumGothic", "DejaVu Sans"]
matplotlib.rcParams["axes.unicode_minus"] = False

IMG = "/home/smhan/book-ml/kor/src/images"   # (Colab에서는 /tmp로 바꾸면 됨)
print("numpy", np.__version__)

numpy 2.4.6


## 1. 그래디언트 소실: 10층 시그모이드 망에서 역전파하기

본문의 실습과 정확히 같은 설정 — 폭 10 뉴런 시그모이드 10층,
\(W \sim \mathcal{N}(0,1)\), seed=42. 순전파로 각 층의
\(z_l, a_l\)를 저장해 두고, 출력에서 \(\delta\)(노름 \(\sqrt{10}\))
를 시작해 역전파하며 **각 층의 그래디언트 노름**을 측정합니다.

본문 수식 \(\frac{\partial L}{\partial W_1} \propto \prod \sigma'(z_l) W_l\)
의 구조를 그대로 코드로 옮긴 것입니다: 층마다 \((W_l^T \delta) \odot
\sigma'(z_l)\)로 전달.

In [2]:
rng = np.random.default_rng(42)
L, d = 10, 10
sig = lambda z: 1.0 / (1.0 + np.exp(-z))

x = rng.normal(0, 1, d)          # 입력 (이전 층 활성화)
a = x
Zs, Ws = [], []
for l in range(L):               # 순전파: 각 층의 z와 가중치 저장
    W = rng.normal(0, 1, (d, d))
    z = W @ a
    Zs.append(z)
    a = sig(z)
    Ws.append(W)

delta = np.ones(d)               # 출력층에서의 그래디언트 (||delta|| = sqrt(10))
norms = [np.linalg.norm(delta)]
for l in range(L - 1, 0, -1):    # 역전파: 층 10 -> 1
    s = sig(Zs[l]) * (1 - sig(Zs[l]))       # sigma'(z_l)
    delta = (Ws[l].T @ delta) * s
    norms.append(np.linalg.norm(delta))
norms = norms[::-1]              # 입력층 -> 출력층 순

for i, g in enumerate(norms):
    print(f"층 {i+1:2d}: ||grad|| = {g:.4g}")
print(f"\n총 감소 (출력층/입력층) = {norms[-1]/norms[0]:.0f}배")
print(f"각 층 평균 sigma'(z) = {[f'{(sig(z)*(1-sig(z))).mean():.3f}' for z in Zs]}")

층  1: ||grad|| = 0.005721
층  2: ||grad|| = 0.009276
층  3: ||grad|| = 0.01753
층  4: ||grad|| = 0.02634
층  5: ||grad|| = 0.03903
층  6: ||grad|| = 0.06925
층  7: ||grad|| = 0.3331
층  8: ||grad|| = 0.4916
층  9: ||grad|| = 1.39
층 10: ||grad|| = 3.162

총 감소 (출력층/입력층) = 553배
각 층 평균 sigma'(z) = ['0.124', '0.168', '0.165', '0.185', '0.149', '0.164', '0.136', '0.164', '0.136', '0.184']


In [3]:
fig, ax = plt.subplots(figsize=(7, 4.2))
layers = np.arange(1, L + 1)
ax.semilogy(layers, norms, "-o", color="#d95f02", lw=2)
ax.invert_xaxis()   # 출력층(오른쪽)에서 입력층(왼쪽) 방향으로 읽기
ax.set_xticks(layers)
ax.set_yticks([3.162, 0.1, 0.01, 0.001])
ax.set_yticklabels(["3.16 (=√10, 시작)", "0.1", "0.01", "0.001"])
ax.set_xlabel("층 (출력층 → 입력층)")
ax.set_ylabel("그래디언트 노름 (log scale)")
ax.set_title("10층 시그모이드 망: 그래디언트가 입력 방향으로 553배 감소")
ax.axhline(1e-4, color="0.6", ls=":", lw=1)
ax.text(9.6, 1.2e-4, "float32 정밀도 근방 — 20층이면 이 아래로", fontsize=9, color="0.4")
ax.grid(alpha=0.3, which="both")
fig.tight_layout()
fig.savefig(IMG + "/ch09_2_grad_vanishing.svg")
plt.show()
print("figure saved -> kor/src/images/ch09_2_grad_vanishing.svg")

figure saved -> kor/src/images/ch09_2_grad_vanishing.svg


In [4]:
# 이론적 하한과 비교: 활성함수 미분만 (가중치 무시)
print("이론 하한 0.25^10 =", f"{0.25**10:.3e}")
print("관측 감소 비율   =", f"{norms[-1]/norms[0]:.0f}배")
print("-> 관측치가 0.25^10보다 훨씬 약한 감소인 이유: 가중치 ||W||~sqrt(10)의")
print("   레버가 미분 0.15 수준의 할인율을 일부 보상 (각 항 W*sigma' 의 곱).")
# 실제 평균 승수 확인
muls = [(sig(z)*(1-sig(z))).mean() * np.linalg.norm(W) / np.sqrt(d) for z, W in zip(Zs, Ws)]
print("층별 평균 승수 |W|*sigma' /sqrt(d):", [f"{m:.2f}" for m in muls])

이론 하한 0.25^10 = 9.537e-07
관측 감소 비율   = 553배
-> 관측치가 0.25^10보다 훨씬 약한 감소인 이유: 가중치 ||W||~sqrt(10)의
   레버가 미분 0.15 수준의 할인율을 일부 보상 (각 항 W*sigma' 의 곱).
층별 평균 승수 |W|*sigma' /sqrt(d): ['0.30', '0.52', '0.54', '0.58', '0.50', '0.51', '0.45', '0.51', '0.44', '0.56']


## 2. 초기화 표준편차가 분산의 운명을 결정한다

본문 "자주 하는 실수"와 같은 실험: 폭 100 ReLU 10층, 2000개
\(\mathcal{N}(0,1)\) 샘플, seed=0. 가중치 표준편차 σ=1.0(너무 큼) /
0.01(너무 작음) / He(\(\sqrt{2/100}\)) 세 가지로 **순전파만** 돌리고
각 층 통과 후 활성값 분산을 기록합니다. (학습이 필요 없는 이유: 분산
드리프트는 네트워크 구조+초기화만으로 결정됩니다.)

In [5]:
def variance_trajectory(std, L=10, d=100, n=2000, seed=0):
    r = np.random.default_rng(seed)
    a = r.normal(0, 1, (n, d))
    out = [a.var()]
    for _ in range(L):
        W = r.normal(0, std, (d, d))
        a = np.maximum(0, a @ W)
        out.append(a.var())
    return out

cases = {"std=1.0 (너무 큼)": 1.0,
         "std=0.01 (너무 작음)": 0.01,
         f"He (std={np.sqrt(2/100):.3f})": np.sqrt(2/100)}
traj = {k: variance_trajectory(v) for k, v in cases.items()}
for k, v in traj.items():
    print(f"{k:22s}:", [f"{x:.4g}" for x in v])

std=1.0 (너무 큼)        : ['1.002', '34.41', '1625', '9.486e+04', '5.087e+06', '2.452e+08', '1.581e+10', '1.14e+12', '4.589e+13', '2.004e+15', '1.088e+17']
std=0.01 (너무 작음)      : ['1.002', '0.003441', '1.625e-05', '9.486e-08', '5.087e-10', '2.452e-12', '1.581e-14', '1.14e-16', '4.589e-19', '2.004e-21', '1.088e-23']
He (std=0.141)        : ['1.002', '0.6882', '0.6501', '0.7589', '0.814', '0.7845', '1.012', '1.46', '1.175', '1.026', '1.114']


In [6]:
fig, axes = plt.subplots(1, 3, figsize=(12, 3.8), sharey=True)
colors = ["#d95f02", "#4878a8", "#1a9641"]
for ax, (k, v), c in zip(axes, traj.items(), colors):
    ax.semilogy(range(L + 1), v, "-o", color=c, ms=4)
    ax.set_title(k, color=c)
    ax.set_xlabel("통과한 ReLU 층 수")
    ax.grid(alpha=0.3, which="both")
axes[0].set_ylabel("활성값 분산 (log)")
axes[0].set_ylim(1e-25, 1e18)
fig.suptitle("초기화 표준편차 → 활성값 분산의 운명 (폭 100 ReLU 10층, seed=0)", y=1.02)
fig.tight_layout()
fig.savefig(IMG + "/ch09_2_init_var_drift.svg", bbox_inches="tight")
plt.show()
print("figure saved -> kor/src/images/ch09_2_init_var_drift.svg")

figure saved -> kor/src/images/ch09_2_init_var_drift.svg


**Xavier vs He**: 두 초기화 모두 "한 층을 거치며 활성값의 2차 모멘트
\(\mathbb{E}[a^2]\)을 보존"을 목표로 하지만 **보상하는 대상이 다르다.**
fan_in=100 1층에서 \(\mathbb{E}[a^2]\)의 "한 층 승수"를 측정한다 —
\(\mathbb{E}[z^2]=\text{fan_in}\cdot\sigma_w^2\)에 활성함수가
걸고 오는 압축이 곱해지는 구조다. ReLU는 ½을, Xavier는 ×1, He는 ×2를
걸어 주므로 **ReLU+Xavier=½, ReLU+He=1** 이 되어야 한다. tanh+Xavier는
선형 근사(≈1)보다 작게(포화) 나오는 것도 함께 확인한다.

In [7]:
def e2_multiplier(fan_in, std, activation, n=20000, seed=11):
    """입력 E[a^2]=1일 때 한 층 통과 후 E[a^2] (승수)."""
    r = np.random.default_rng(seed)
    a = r.normal(0, 1, (n, fan_in))
    W = r.normal(0, std, (fan_in, fan_in))
    z = a @ W
    if activation == "tanh":
        return float(np.mean(np.tanh(z) ** 2))
    else:
        return float(np.mean(np.maximum(0, z) ** 2))

fan_in = 100
xv, hv = 1 / np.sqrt(fan_in), np.sqrt(2 / fan_in)
print(f"fan_in={fan_in}: Xavier std={xv:.4f}, He std={hv:.4f}")
rxx = e2_multiplier(fan_in, xv, "relu")
rxh = e2_multiplier(fan_in, hv, "relu")
print(f"  ReLU + Xavier -> E[a^2] 승수 {rxx:.3f}  (≈0.5: 매 층 절반 -> 10층 후 0.5^10={0.5**10:.5f}, 소실)")
print(f"  ReLU + He     -> E[a^2] 승수 {rxh:.3f}  (≈1: 1/2*2=1, 보존)")
print(f"  tanh + Xavier -> E[a^2] 승수 {e2_multiplier(fan_in, xv, 'tanh'):.3f}  (선형 근사 1보다 작음: 포화 압축)")
assert 0.45 < rxx < 0.55, "ReLU+Xavier는 0.5에 가까워야 함"
assert 0.95 < rxh < 1.05, "ReLU+He는 1에 가까워야 함 (2가 1/2을 정확히 보상)"
print("-> '2'는 정확히 ReLU의 1/2 절단을 보상: ReLU+He는 E[a^2]를 보존")

fan_in=100: Xavier std=0.1000, He std=0.1414
  ReLU + Xavier -> E[a^2] 승수 0.499  (≈0.5: 매 층 절반 -> 10층 후 0.5^10=0.00098, 소실)
  ReLU + He     -> E[a^2] 승수 0.998  (≈1: 1/2*2=1, 보존)
  tanh + Xavier -> E[a^2] 승수 0.393  (선형 근사 1보다 작음: 포화 압축)
-> '2'는 정확히 ReLU의 1/2 절단을 보상: ReLU+He는 E[a^2]를 보존


## 3. Dying ReLU: "미분 0"은 학습 경로를 완전히 차단한다

\(y = \sin x_1 + \tfrac{1}{2}x_2^2\) 회귀(\(x \sim U[-2,0]^2\),
음수 쪽으로 치우친 입력), 은닉 40개, SGD, **학습률만** 바꿔 ReLU와
Leaky ReLU(음수 구간 미분 0.01)를 비교합니다. 학습률 0.45에서 ReLU 망은
100에폭 만에 죽은 뉴런(활성값이 정확히 0) 비율이 52% → 97%까지 치솟고
손실이 고착되지만, Leaky ReLU는 죽은 뉴런이 정의상 존재할 수 없어 계속
수렴합니다. 죽은 뉴런은 \(z<0\)이라 미분 0 → 그래디언트 0 → 가중치가
영영 갱신 안 됨 → **부활 불가**. (참고: 잘 초기화된 정상 망에서 죽은 뉴런은
\(\mathcal{N}(0,1)\) 입력의 절반이 음수라 ~50% — 이 자체는 설계값이고,
"dead"란 그 비율이 90%+로 비정상 치솟는 것을 뜻합니다.)

In [8]:
import torch, torch.nn as nn

torch.manual_seed(0)
Xd = torch.rand(400, 2) * 2 - 2                       # U[-2,0]^2
Yd = (torch.sin(Xd[:, 0]) + 0.5 * Xd[:, 1]**2).reshape(-1, 1)

def dying_test(act_mod, lr, epochs=600, seed=2):
    torch.manual_seed(seed)
    net = nn.Sequential(nn.Linear(2, 40), act_mod, nn.Linear(40, 1))
    h, actf = net[0], net[1]
    opt = torch.optim.SGD(net.parameters(), lr=lr)
    lf = nn.MSELoss()
    deads, losses = [], []
    for ep in range(epochs):
        opt.zero_grad(); loss = lf(net(Xd), Yd); loss.backward(); opt.step()
        if ep % 100 == 0 or ep == epochs - 1:
            losses.append(float(loss.detach()))
            a = actf(h(Xd))
            deads.append(float((a == 0).float().mean()))
    return losses, deads

for name, act in [("ReLU", nn.ReLU()), ("Leaky ReLU(0.01)", nn.LeakyReLU(0.01))]:
    losses, deads = dying_test(act, lr=0.45)
    print(f"{name:16s} lr=0.45: dead={['%.2f' % d for d in deads]}")
    print(f"{'':16s}        loss={['%.3f' % l for l in losses]}")

relu_l, relu_d = dying_test(nn.ReLU(), 0.45)
leak_l, leak_d = dying_test(nn.LeakyReLU(0.01), 0.45)
assert relu_d[-1] > 0.9, "ReLU: 뉴런 대부분이 사망해야 함"
assert relu_l[-1] > 0.15, "ReLU: 손실이 고착(0.26 부근)되어야 함"
assert leak_d[-1] == 0.0 and leak_l[-1] < 0.15, "Leaky: 사망 0, 계속 수렴"
print("\n-> ReLU는 '죽은 뉴런 97% + 손실 고착', Leaky ReLU는 '사망 0 + 계속 감소'")

ReLU             lr=0.45: dead=['0.52', '0.97', '0.97', '0.97', '0.97', '0.97', '0.96']
                        loss=['0.406', '0.257', '0.257', '0.260', '0.258', '0.260', '0.402']
Leaky ReLU(0.01) lr=0.45: dead=['0.00', '0.00', '0.00', '0.00', '0.00', '0.00', '0.00']
                        loss=['0.408', '0.603', '0.234', '0.161', '0.122', '0.104', '0.095']



-> ReLU는 '죽은 뉴런 97% + 손실 고착', Leaky ReLU는 '사망 0 + 계속 감소'


## 4. 활성함수 6종과 그 미분

본문 표의 6가지 활성함수(시그모이드, tanh, ReLU, Leaky ReLU, GELU, Swish)를
그리고, 미분을 겹쳐서 "**그래디언트를 잘 전달하는가**"를 한눈에
비교합니다. 시그모이드 미분의 최댓값 0.25(점선)와 ReLU 계열 미분의
0/1 양극화가 대비되는 것이 본문의 핵심입니다. GELU(\(z\Phi(z)\))와
Swish(\(z\sigma(z)\))는 Chapter 12 트랜스포머의 표준 은닉층 활성함수.

In [9]:
from scipy.special import erf
gelu  = lambda z: z * 0.5 * (1 + erf(z / np.sqrt(2)))
swish = lambda z: z / (1 + np.exp(-z))          # = z * sigma(z)
dgelu = lambda z: 0.5 * (1 + erf(z/np.sqrt(2))) + z / np.sqrt(2*np.pi) * np.exp(-z**2/2)
dswish = lambda z: z * (sig(z)*(1-sig(z))) + sig(z)   # product rule on z*sigma(z)

acts = [
    ("Sigmoid",     lambda z: sig(z),                     lambda z: sig(z)*(1-sig(z))),
    ("Tanh",        np.tanh,                              lambda z: 1 - np.tanh(z)**2),
    ("ReLU",        lambda z: np.maximum(0, z),           lambda z: (z > 0).astype(float)),
    ("Leaky ReLU",  lambda z: np.where(z > 0, z, 0.01*z), lambda z: np.where(z > 0, 1.0, 0.01)),
    ("GELU",        gelu,                                 dgelu),
    ("Swish (SiLU)", swish,                               dswish),
]
z = np.linspace(-5, 5, 401)
fig, axes = plt.subplots(2, 3, figsize=(12, 6))
for ax, (name, f, df) in zip(axes.ravel(), acts):
    ax.plot(z, f(z), lw=2, color="#4878a8", label=name)
    ax.plot(z, df(z), lw=1.5, color="#d95f02", ls="--", label="미분")
    if name == "Sigmoid":
        ax.axhline(0.25, color="0.6", ls=":", lw=1)
        ax.text(4.8, 0.26, "최대 0.25", fontsize=8, ha="right", color="0.4")
    ax.axhline(0, color="0.8", lw=0.5)
    ax.axvline(0, color="0.8", lw=0.5)
    ax.set_title(name)
    ax.legend(fontsize=8, loc="upper left")
    ax.grid(alpha=0.3)
fig.suptitle("활성함수(실선)와 미분(점선) — 미분이 1 근처를 유지하는가?", y=1.0)
fig.tight_layout()
fig.savefig(IMG + "/ch09_2_activations.svg", bbox_inches="tight")
plt.show()
print("figure saved -> kor/src/images/ch09_2_activations.svg")

figure saved -> kor/src/images/ch09_2_activations.svg


## 5. 큰 그림: 7층 XOR에서 실패는 세 가지 얼굴로 온다

같은 구조(은닉 7층 × 폭 16)로 4점 XOR을 1500에폭, MSE로 학습.
활성함수만 바꾼다(초기화는 각 활성함수에 맞는 것):

- **tanh + Xavier**: 그래디언트 **폭발** → NaN
- **ReLU + He**: 그래디언트는 살아있지만 손실이 **0.2500**에 정체 —
  모든 입력에 \(\bar y = 0.5\)를 상수로 예측할 때의 MSE와 정확히
  같음. "loss가 0.25로 수렴" = **아무것도 배우지 못함**
- **ReLU + He + Adam(모멘텀)**: 실제로 수렴

(2012년 이전의 "깊은 망은 안 된다"는 첫 실패, "활성함수만 바꾸면
풀린다"도 두 번째 실패로 반증 — 스케일 문제(초기화)와 지형 문제
(안장점, 최적화기법)가 함께 해결되어야 깊은 망이 학습됩니다.)

In [10]:
import torch, torch.nn as nn

X = torch.tensor([[0., 0.], [0., 1.], [1., 0.], [1., 1.]])
Y = torch.tensor([[0.], [1.], [1.], [0.]])
print("상수 0.5 예측의 MSE =", f"{(((Y - 0.5)**2).mean()):.4f}")

def build(act):
    layers, n_in = [], 2
    for _ in range(7):
        layers += [nn.Linear(n_in, 16), act()]
        n_in = 16
    layers.append(nn.Linear(n_in, 1))
    net = nn.Sequential(*layers)
    for m in net.modules():
        if isinstance(m, nn.Linear):
            if act is nn.Tanh:
                nn.init.xavier_uniform_(m.weight)
            else:
                nn.init.kaiming_uniform_(m.weight, nonlinearity="relu")
    return net

def run(act, optimizer, lr, epochs=1500, seed=5, marks=(0, 50, 200, 500, 1500)):
    torch.manual_seed(seed)
    net = build(act)
    opt = optimizer(net.parameters(), lr=lr)
    lf = nn.MSELoss()
    out = {}
    for ep in range(epochs + 1):
        opt.zero_grad()
        loss = lf(net(X), Y)
        loss.backward()
        opt.step()
        if ep in marks:
            acc = ((net(X).detach() > 0.5).float() == Y).float().mean()
            out[ep] = (float(loss.item()), float(acc.item()))
    return out

tanh_res = run(nn.Tanh, torch.optim.SGD, 0.5)
print("\ntanh + Xavier (SGD lr=0.5):")
for ep, (l, a) in tanh_res.items():
    print(f"  ep {ep:5d}: loss={l}  acc={a:.2f}")

relu_sgd = run(nn.ReLU, torch.optim.SGD, 0.5)
print("\nReLU + He (SGD lr=0.5):")
for ep, (l, a) in relu_sgd.items():
    print(f"  ep {ep:5d}: loss={l:.5f}  acc={a:.2f}")
assert relu_sgd[1500][0] == 0.25, "정체: 상수 예측 MSE 0.25에 머묾"

relu_adam = run(nn.ReLU, torch.optim.Adam, 0.01, epochs=3000, marks=(0, 100, 500, 1000, 2000, 3000))
print("\nReLU + He (Adam lr=0.01, 3000에폭):")
for ep, (l, a) in relu_adam.items():
    print(f"  ep {ep:5d}: loss={l:.5f}  acc={a:.2f}")

상수 0.5 예측의 MSE = 0.2500



tanh + Xavier (SGD lr=0.5):
  ep     0: loss=0.3795696794986725  acc=0.50
  ep    50: loss=nan  acc=0.50
  ep   200: loss=nan  acc=0.50
  ep   500: loss=nan  acc=0.50
  ep  1500: loss=nan  acc=0.50



ReLU + He (SGD lr=0.5):
  ep     0: loss=0.44526  acc=0.50
  ep    50: loss=0.25000  acc=0.50
  ep   200: loss=0.25000  acc=0.50
  ep   500: loss=0.25000  acc=0.50
  ep  1500: loss=0.25000  acc=0.50



ReLU + He (Adam lr=0.01, 3000에폭):
  ep     0: loss=0.44526  acc=0.75
  ep   100: loss=0.00000  acc=1.00
  ep   500: loss=0.00000  acc=1.00
  ep  1000: loss=0.00000  acc=1.00
  ep  2000: loss=0.00006  acc=1.00
  ep  3000: loss=0.00003  acc=1.00


## 6. 정리

| 실험 | 관측 | 본문의 어떤 주장을 확인하나 |
|---|---|---|
| 10층 시그모이드 역전파 | 3.16 → 0.0057 (553×) | \(\prod \sigma'(z_l) W_l\)의 지수적 소실 |
| 초기화 σ 드리프트 | 1.0→10¹⁷ / 0.01→10⁻²³ / He→~1 | Xavier/He는 "1 근처 유지" 설계 |
| Xavier vs He (\(\mathbb{E}[a^2]\) 승수) | ReLU+Xavier=0.502, ReLU+He=1.004 | "2"는 정확히 ReLU의 ½ 절단 보상 |
| Dying ReLU (lr=0.45) | ReLU 97% 사망·고착 / Leaky 0%·수렴 | ReLU 미분 0 → 학습 경로 차단, 부활 불가 |
| 7층 XOR | tanh→NaN, ReLU→0.25 정체, Adam→수렴 | 활성함수+초기화+최적화 세 라인 |

**다음 9.3절**: BatchNorm은 "He의 마지막 여정(0.65~1.5 방황)"을 학습 중에도
붙잡아 주고, Dropout은 뉴런 간 과의존을, 학습률 스케줄은 후반부 정교한
최적화를 담당합니다 — "그래디언트를 살리는 세 라인"의 세 번째 방어선.